In [1]:
!pip install tensorflow opencv-python matplotlib

In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import os

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/pcos/processed_img_0_1071.jpg'
output_path = '/kaggle/working/gradcam_output1001new.jpg'

# Load model
model = load_model(model_path)

# Identify the last conv layer
for layer in model.layers[::-1]:
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

# Preprocess the image
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array / 255.0

# Predict the class of the image
class_names = ['pcos', 'normal']  # Update based on your model classes
img_array = get_img_array(img_path, (224, 224))
preds = model.predict(img_array)

# Get predicted class and confidence for all classes
predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]
confidence = preds[0][predicted_class_index]

# Print predicted class and confidence for all classes
print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# Generate Grad-CAM heatmap
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Overlay heatmap on original image with label
def save_and_display_gradcam(img_path, heatmap, output_path, label, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = heatmap_color * alpha + img
    superimposed_img = np.uint8(superimposed_img)
    
    # Add text label
    cv2.putText(superimposed_img, f"Pred: {label}", (10, 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imwrite(output_path, superimposed_img)

# Run Grad-CAM and save output
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
save_and_display_gradcam(img_path, heatmap, output_path, predicted_class)

print("✅ Grad-CAM saved at:", output_path)


2025-06-09 05:56:16.367110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749448576.763162      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749448576.863492      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I0000 00:00:1749448596.246487      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749448596.247133      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability:

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step


I0000 00:00:1749448613.684228      95 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


🔍 Predicted Class and Confidence Levels:
pcos: 0.98 confidence
normal: 0.02 confidence
✅ Grad-CAM saved at: /kaggle/working/gradcam_output1001new.jpg


In [3]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import os

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/normal/processed_img_0_100.jpg'
output_path = '/kaggle/working/NewImageNORMAL.jpg'

# Load model
model = load_model(model_path)

# Identify the last conv layer
for layer in model.layers[::-1]:
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

# Preprocess the image
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array / 255.0

# Predict the class of the image
class_names = ['normal', 'pcos']  # Update based on your model classes
img_array = get_img_array(img_path, (224, 224))
preds = model.predict(img_array)

# Get predicted class and confidence for all classes
predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]
confidence = preds[0][predicted_class_index]

# Print predicted class and confidence for all classes
print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# Generate Grad-CAM heatmap
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Overlay heatmap on original image with label
def save_and_display_gradcam(img_path, heatmap, output_path, label, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = heatmap_color * alpha + img
    superimposed_img = np.uint8(superimposed_img)
    
    # Add text label
    cv2.putText(superimposed_img, f"Pred: {label}", (10, 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imwrite(output_path, superimposed_img)

# Run Grad-CAM and save output
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
save_and_display_gradcam(img_path, heatmap, output_path, predicted_class)

print("✅ Grad-CAM saved at:", output_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
🔍 Predicted Class and Confidence Levels:
normal: 0.99 confidence
pcos: 0.01 confidence
✅ Grad-CAM saved at: /kaggle/working/NewImageNORMAL.jpg


In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/normal/processed_img_0_100.jpg'
output_path = '/kaggle/working/gradcampp_PCOS.jpg'

# Load model
model = load_model(model_path)

# Identify the last conv layer
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

# Preprocess the image
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array / 255.0

# Predict the class
class_names = ['pcos', 'normal']
img_array = get_img_array(img_path, (224, 224))
preds = model.predict(img_array)

predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]
confidence = preds[0][predicted_class_index]

print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# Grad-CAM++ function
def make_gradcampp_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape(persistent=True) as tape1:
        with tf.GradientTape(persistent=True) as tape2:
            with tf.GradientTape() as tape3:
                conv_outputs, predictions = grad_model(img_array)
                if pred_index is None:
                    pred_index = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]

            grads = tape3.gradient(class_channel, conv_outputs)  # first order grads
        grads2 = tape2.gradient(grads, conv_outputs)  # second order grads
    grads3 = tape1.gradient(grads2, conv_outputs)  # third order grads

    conv_outputs = conv_outputs[0]
    grads = grads[0]
    grads2 = grads2[0]
    grads3 = grads3[0]

    numerator = grads2
    denominator = 2 * grads2 + grads3 * conv_outputs
    denominator = tf.where(denominator != 0.0, denominator, tf.ones_like(denominator))
    alphas = numerator / denominator

    weights = tf.maximum(grads, 0.0)
    alphas_thresholding = tf.where(weights > 0, alphas, tf.zeros_like(alphas))

    deep_linearization_weights = tf.reduce_sum(alphas_thresholding * weights, axis=(0, 1))

    heatmap = tf.reduce_sum(deep_linearization_weights * conv_outputs, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    heatmap = heatmap / (max_val + 1e-10)

    return heatmap.numpy()

# Overlay heatmap on original image
def save_and_display_gradcam(img_path, heatmap, output_path, label, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(heatmap_color, alpha, img, 1 - alpha, 0)
    
    # Add label text
    cv2.putText(superimposed_img, f"Pred: {label}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imwrite(output_path, superimposed_img)
    # Optionally show image with matplotlib (uncomment if needed)
    # plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # plt.show()

# Run Grad-CAM++
heatmap = make_gradcampp_heatmap(img_array, model, last_conv_layer_name, predicted_class_index)
save_and_display_gradcam(img_path, heatmap, output_path, predicted_class)

print("✅ Grad-CAM++ saved at:", output_path)


2025-06-09 06:06:05.592376: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749449166.072912      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749449166.244164      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I0000 00:00:1749449186.819586      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749449186.820271      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability:

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step


I0000 00:00:1749449204.558172      98 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


🔍 Predicted Class and Confidence Levels:
pcos: 0.99 confidence
normal: 0.01 confidence
✅ Grad-CAM++ saved at: /kaggle/working/gradcampp_PCOS.jpg


In [3]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/normal/processed_img_0_100.jpg'
output_path = '/kaggle/working/lime_PCOS.jpg'

# Load model
model = load_model(model_path)

# Class names
class_names = ['normal', 'pcos']

# Preprocess image
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    return array

# Prediction function for LIME
def predict_fn(images):
    # Normalize images to [0,1]
    images = images.astype(np.float32) / 255.0
    return model.predict(images)

# Load and preprocess image
img = get_img_array(img_path, (224, 224))
img_for_model = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)

# Predict class
preds = model.predict(img_for_model)
predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]

print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# Initialize LIME image explainer
explainer = lime_image.LimeImageExplainer()

# Explain prediction with LIME
explanation = explainer.explain_instance(
    img.astype(np.double),
    predict_fn,
    top_labels=2,
    hide_color=0,
    num_samples=1000  # Number of perturbed samples generated
)

# Get image and mask for the predicted class
temp, mask = explanation.get_image_and_mask(
    label=predicted_class_index,
    positive_only=True,
    hide_rest=False,
    num_features=10,
    min_weight=0.05
)

# Mark boundaries on the original image
img_boundry = mark_boundaries(temp / 255.0, mask)

# Save explanation image
plt.figure(figsize=(6,6))
plt.title(f'LIME Explanation for class: {predicted_class}')
plt.axis('off')
plt.imshow(img_boundry)
plt.savefig(output_path, bbox_inches='tight')
plt.close()

print("✅ LIME explanation saved at:", output_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
🔍 Predicted Class and Confidence Levels:
normal: 0.99 confidence
pcos: 0.01 confidence


  0%|          | 0/1000 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━

In [5]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/normal/processed_img_0_100.jpg'
output_path = '/kaggle/working/lime_PCOS.jpg'

# Load model
model = load_model(model_path)

# Class names (in same order as model predicts)
class_names = ['normal', 'pcos']  # Adjust this depending on your model's training order

# Preprocess image for model
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    return array.astype(np.uint8)

# Prediction function for LIME
def predict_fn(images):
    images = np.array(images).astype(np.float32) / 255.0  # Normalize
    return model.predict(images)

# Load image
img = get_img_array(img_path, (224, 224))
img_model_input = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)

# Model prediction
preds = model.predict(img_model_input)
predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]

print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# LIME explanation
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    img,                # Unnormalized uint8 image
    predict_fn,         # Model prediction function
    top_labels=2,       # Get top 2 class explanations
    hide_color=0, 
    num_samples=1000    # Number of perturbed samples
)

# Get explanation image & mask
temp, mask = explanation.get_image_and_mask(
    label=predicted_class_index,
    positive_only=True,
    hide_rest=False,
    num_features=10,
    min_weight=0.05
)

# Mark boundaries and save
img_boundary = mark_boundaries(temp / 255.0, mask)
plt.figure(figsize=(6,6))
plt.title(f'LIME Explanation for class: {predicted_class}')
plt.axis('off')
plt.imshow(img_boundary)
plt.savefig(output_path, bbox_inches='tight')
plt.close()

print("✅ LIME explanation saved at:", output_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
🔍 Predicted Class and Confidence Levels:
normal: 0.99 confidence
pcos: 0.01 confidence


  0%|          | 0/1000 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━

In [7]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Paths
model_path = '/kaggle/input/modelpcosnew/tensorflow2/default/1/fullModel.h5'
img_path = '/kaggle/input/newsplit2augpcos/NewAugSpliT2split2000pcos/val/pcos/processed_img_0_1071.jpg'
output_path = '/kaggle/working/lime_PCOS2.jpg'

# Load model
model = load_model(model_path)

# Class names (in same order as model predicts)
class_names = ['normal', 'pcos']  # Adjust this depending on your model's training order

# Preprocess image for model
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    return array.astype(np.uint8)

# Prediction function for LIME
def predict_fn(images):
    images = np.array(images).astype(np.float32) / 255.0  # Normalize
    return model.predict(images)

# Load image
img = get_img_array(img_path, (224, 224))
img_model_input = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)

# Model prediction
preds = model.predict(img_model_input)
predicted_class_index = np.argmax(preds[0])
predicted_class = class_names[predicted_class_index]

print("🔍 Predicted Class and Confidence Levels:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {preds[0][i]:.2f} confidence")

# LIME explanation
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    img,                # Unnormalized uint8 image
    predict_fn,         # Model prediction function
    top_labels=2,       # Get top 2 class explanations
    hide_color=0, 
    num_samples=1000    # Number of perturbed samples
)

# Get explanation image & mask
temp, mask = explanation.get_image_and_mask(
    label=predicted_class_index,
    positive_only=True,
    hide_rest=False,
    num_features=10,
    min_weight=0.05
)

# Mark boundaries and save
img_boundary = mark_boundaries(temp / 255.0, mask)
plt.figure(figsize=(6,6))
plt.title(f'LIME Explanation for class: {predicted_class}')
plt.axis('off')
plt.imshow(img_boundary)
plt.savefig(output_path, bbox_inches='tight')
plt.close()

print("✅ LIME explanation saved at:", output_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
🔍 Predicted Class and Confidence Levels:
normal: 0.98 confidence
pcos: 0.02 confidence


  0%|          | 0/1000 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━